# Портфель EW из победителей SPA

Создает портфели с равным весом на основе предварительно рассчитанных победителей SPA (`lam=1.0`, `Base`)

Сравнение и устойчивость для `1d U17` и `4h U12`.


### Setup + Config

In [ ]:
from __future__ import annotations

import ast
import logging
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

from src.artifacts import safe_write_parquet
from src.metrics import compute_metrics_from_returns
from src import report as report_mod

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("portfolio_nb")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Cannot locate project root from cwd={Path.cwd()}")

RESULTS_DIR = PROJECT_ROOT / "results" / "portfolio"
FIGURES_DIR = PROJECT_ROOT / "figures" / "portfolio"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "winners": PROJECT_ROOT / "results" / "portfolio_prepare" / "spa_winners_lam1_base.parquet",
    "base_1d_returns": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "returns_oos.parquet",
    "base_1d_bench": PROJECT_ROOT / "results" / "runner_exp" / "1d" / "bench_returns_oos.parquet",
    "onchain_1d_returns": PROJECT_ROOT / "results" / "runner_onchain" / "returns_oos.parquet",
    "base_4h_returns": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "returns_oos.parquet",
    "base_4h_bench": PROJECT_ROOT / "results" / "runner_exp" / "4h" / "bench_returns_oos.parquet",
}
for name, path in INPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name} -> {path}")

LAMBDA = 1.0
BASE_COST = "Base"
COSTS = ["Low", "Base", "High"]
PPY = {"1d": 365, "4h": 2190}
MIN_BARS = {"1d": 50, "4h": 300}

SUBPERIODS = {
    "Bull 2021": ("2021-01-01", "2021-11-01"),
    "Bear 2022": ("2022-01-01", "2023-01-01"),
    "Recovery 2023": ("2023-01-01", "2024-01-01"),
    "Bull 2024": ("2024-01-01", "2025-01-01"),
    "H1 2025": ("2025-01-01", "2025-09-01"),
    "H2 2025+": ("2025-09-01", "2026-04-01"),
}

METRIC_COLS = ["Total Return", "CAGR", "Ann. Vol", "Sharpe", "Sortino", "MaxDD", "Calmar", "CDaR_95", "n_bars"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


PROJECT_ROOT: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
RESULTS_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\portfolio
FIGURES_DIR: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\figures\portfolio


### Построение Universe

In [ ]:
def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if "date" not in df.columns:
        if "datetime_utc" in df.columns:
            df = df.rename(columns={"datetime_utc": "date"})
        elif "index" in df.columns:
            df = df.rename(columns={"index": "date"})
        else:
            raise KeyError(f"No date-like column found. Columns: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def assert_unique_key(df: pd.DataFrame, *, name: str) -> None:
    key = ["date", "symbol", "cost", "strategy_id"]
    dups = int(df.duplicated(subset=key).sum())
    if dups > 0:
        raise RuntimeError(f"{name}: duplicate rows by {key}: {dups}")


def parse_winners_cell(x) -> list[str]:
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [str(v) for v in list(x)]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            y = ast.literal_eval(s)
            if isinstance(y, (list, tuple, set)):
                return [str(v) for v in list(y)]
        except Exception:
            pass
    return [s] if s else []


def series_from_long(df_long: pd.DataFrame, *, symbol: str, cost: str, strategy_id: str) -> pd.Series:
    g = df_long[
        (df_long["symbol"] == symbol)
        & (df_long["cost"] == cost)
        & (df_long["strategy_id"] == strategy_id)
    ].copy()
    if g.empty:
        return pd.Series(dtype=float)
    g = g.sort_values("date")
    return g.set_index("date")["ret"].astype(float)


def wide_returns(df_long: pd.DataFrame, *, symbol: str, cost: str, strategy_ids: list[str]) -> pd.DataFrame:
    x = df_long[
        (df_long["symbol"] == symbol)
        & (df_long["cost"] == cost)
        & (df_long["strategy_id"].astype(str).isin(list(map(str, strategy_ids))))
    ].copy()
    if x.empty:
        return pd.DataFrame()
    w = x.pivot_table(index="date", columns="strategy_id", values="ret", aggfunc="mean").sort_index()
    cols = [sid for sid in strategy_ids if sid in w.columns]
    if not cols:
        return pd.DataFrame(index=w.index)
    return w[cols].astype(float)


winners_raw = pd.read_parquet(INPUTS["winners"])
base_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["base_1d_returns"]))
base_1d_bench = ensure_date_col(pd.read_parquet(INPUTS["base_1d_bench"]))
onchain_1d_returns = ensure_date_col(pd.read_parquet(INPUTS["onchain_1d_returns"]))
base_4h_returns = ensure_date_col(pd.read_parquet(INPUTS["base_4h_returns"]))
base_4h_bench = ensure_date_col(pd.read_parquet(INPUTS["base_4h_bench"]))

for name, df in [
    ("base_1d_returns", base_1d_returns),
    ("base_1d_bench", base_1d_bench),
    ("onchain_1d_returns", onchain_1d_returns),
    ("base_4h_returns", base_4h_returns),
    ("base_4h_bench", base_4h_bench),
]:
    assert_unique_key(df, name=name)

returns_1d_u17 = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
assert_unique_key(returns_1d_u17, name="returns_1d_u17")
returns_4h_u12 = base_4h_returns.copy()

UNIVERSES = {
    "1d_u17": {
        "timeframe": "1d",
        "scenario": "u17",
        "returns": returns_1d_u17,
        "bench": base_1d_bench,
    },
    "4h_u12": {
        "timeframe": "4h",
        "scenario": "u12",
        "returns": returns_4h_u12,
        "bench": base_4h_bench,
    },
}

winners_df = winners_raw.copy()
winners_df = winners_df[winners_df["cost"].astype(str) == BASE_COST].copy()
if "lam" in winners_df.columns:
    winners_df = winners_df[np.isclose(winners_df["lam"].astype(float), float(LAMBDA), atol=1e-12)].copy()

winners_df["timeframe"] = winners_df["timeframe"].astype(str)
winners_df["scenario"] = winners_df["scenario"].astype(str)
winners_df["symbol"] = winners_df["symbol"].astype(str)
winners_df["winners"] = winners_df["winners"].apply(parse_winners_cell)
winners_df["n_winners"] = winners_df["winners"].apply(len)

expected_cases = {("1d", "u17", s) for s in ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]}
expected_cases |= {("4h", "u12", s) for s in ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "DOGEUSDT"]}
actual_cases = set(zip(winners_df["timeframe"], winners_df["scenario"], winners_df["symbol"]))
missing = sorted(expected_cases - actual_cases)
if missing:
    raise AssertionError(f"Missing winners cases: {missing}")

if (winners_df["n_winners"] < 1).any():
    bad = winners_df[winners_df["n_winners"] < 1][["timeframe", "scenario", "symbol"]]
    raise AssertionError(f"Found empty winners list for cases: {bad.to_dict('records')}")

case_summary = winners_df[["timeframe", "scenario", "symbol", "n_winners"]].sort_values(["timeframe", "symbol"]).reset_index(drop=True)
display(case_summary)


,timeframe,scenario,symbol,n_winners
0,1d,u17,BNBUSDT,9
1,1d,u17,BTCUSDT,17
2,1d,u17,DOGEUSDT,14
3,1d,u17,ETHUSDT,16
4,1d,u17,XRPUSDT,14
5,4h,u12,BNBUSDT,12
6,4h,u12,BTCUSDT,12
7,4h,u12,DOGEUSDT,12
8,4h,u12,ETHUSDT,12
9,4h,u12,XRPUSDT,12


### Returns портфеля

In [ ]:
membership_rows = []
port_ret_parts = []

for r in winners_df.itertuples(index=False):
    tf = str(r.timeframe)
    sc = str(r.scenario)
    sym = str(r.symbol)
    winners = list(map(str, list(r.winners)))
    u = UNIVERSES[f"{tf}_{sc}"]

    wide = wide_returns(u["returns"], symbol=sym, cost=BASE_COST, strategy_ids=winners)
    if wide.empty:
        raise RuntimeError(f"No returns found for winners: tf={tf}, sc={sc}, symbol={sym}")

    used_winners = [sid for sid in winners if sid in wide.columns]
    if not used_winners:
        raise RuntimeError(f"No winner columns present in wide matrix: tf={tf}, symbol={sym}")

    wide = wide[used_winners].fillna(0.0)
    n = len(used_winners)
    weights = np.ones(n, dtype=float) / float(n)
    port_series = (wide * weights).sum(axis=1)
    port_series.name = "ret"

    part = port_series.reset_index().rename(columns={"index": "date"})
    part["timeframe"] = tf
    part["scenario"] = sc
    part["symbol"] = sym
    part["cost"] = BASE_COST
    part["lam"] = float(LAMBDA)
    part["strategy_id"] = "PORT:EW_SPA_WINNERS"
    port_ret_parts.append(part)

    membership_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost_selection": BASE_COST,
            "lam": float(LAMBDA),
            "n_winners": int(n),
            "weights_rule": "EW_1_over_N",
            "winners": used_winners,
        }
    )

portfolio_returns = pd.concat(port_ret_parts, ignore_index=True).sort_values(["timeframe", "symbol", "date"])
portfolio_membership = pd.DataFrame(membership_rows).sort_values(["timeframe", "symbol"]).reset_index(drop=True)

safe_write_parquet(portfolio_returns, RESULTS_DIR / "portfolio_returns_ew_lam1_base.parquet")
safe_write_parquet(portfolio_membership, RESULTS_DIR / "portfolio_membership_lam1_base.parquet")

# мат sanity
first = portfolio_membership.iloc[0]
u0 = UNIVERSES[f"{first.timeframe}_{first.scenario}"]
wide0 = wide_returns(u0["returns"], symbol=first.symbol, cost=BASE_COST, strategy_ids=list(first.winners)).fillna(0.0)
w0 = np.ones(wide0.shape[1], dtype=float) / float(wide0.shape[1])
ref0 = (wide0 * w0).sum(axis=1)
chk0 = portfolio_returns[(portfolio_returns["timeframe"] == first.timeframe) & (portfolio_returns["symbol"] == first.symbol)].set_index("date")["ret"]
ref0, chk0 = ref0.align(chk0, join="inner")
if not np.allclose(ref0.values, chk0.values, atol=1e-12, rtol=0.0):
    raise AssertionError("Portfolio EW sanity check failed on first case.")

print("portfolio_returns:", portfolio_returns.shape)
print("portfolio_membership:", portfolio_membership.shape)
display(portfolio_membership)


portfolio_returns: (61036, 8)
portfolio_membership: (10, 8)


,timeframe,scenario,symbol,cost_selection,lam,n_winners,weights_rule,winners
0,1d,u17,BNBUSDT,Base,1.0,9,EW_1_over_N,"[R1:RSI_MR, R2:Bollinger_MR, R3:ZScore_MR, S1:..."
1,1d,u17,BTCUSDT,Base,1.0,17,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
2,1d,u17,DOGEUSDT,Base,1.0,14,EW_1_over_N,"[R1:RSI_MR, R1OC:RSI_MR_Onchain, R2OC:Bollinge..."
3,1d,u17,ETHUSDT,Base,1.0,16,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
4,1d,u17,XRPUSDT,Base,1.0,14,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R1OC:RSI_MR_Onchain, R2:..."
5,4h,u12,BNBUSDT,Base,1.0,12,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
6,4h,u12,BTCUSDT,Base,1.0,12,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
7,4h,u12,DOGEUSDT,Base,1.0,12,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
8,4h,u12,ETHUSDT,Base,1.0,12,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."
9,4h,u12,XRPUSDT,Base,1.0,12,EW_1_over_N,"[M1:TSMOM, R1:RSI_MR, R2:Bollinger_MR, R3:ZSco..."


### Метрики и сравнение с бенчмарком

In [ ]:
def metrics_as_row(returns: pd.Series, *, periods_per_year: int) -> dict:
    m = compute_metrics_from_returns(returns.astype(float), periods_per_year=periods_per_year)
    out = {k: m.get(k, np.nan) for k in METRIC_COLS if k in m}
    out["n_bars"] = int(m.get("n_bars", len(returns)))
    return out


def pairwise_corr_mean(wide: pd.DataFrame) -> float:
    if wide.shape[1] < 2:
        return float("nan")
    c = wide.corr().to_numpy(dtype=float)
    iu = np.triu_indices_from(c, k=1)
    vals = c[iu]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return float("nan")
    return float(np.mean(vals))


comparison_rows = []
diag_rows = []
case_meta_rows = []

for r in portfolio_membership.itertuples(index=False):
    tf = str(r.timeframe)
    sc = str(r.scenario)
    sym = str(r.symbol)
    winners = list(map(str, list(r.winners)))
    ppy = int(PPY[tf])

    u = UNIVERSES[f"{tf}_{sc}"]
    winners_wide = wide_returns(u["returns"], symbol=sym, cost=BASE_COST, strategy_ids=winners).fillna(0.0)

    port = portfolio_returns[
        (portfolio_returns["timeframe"] == tf)
        & (portfolio_returns["scenario"] == sc)
        & (portfolio_returns["symbol"] == sym)
        & (portfolio_returns["cost"] == BASE_COST)
    ].set_index("date")["ret"].astype(float)

    bench = series_from_long(u["bench"], symbol=sym, cost=BASE_COST, strategy_id="BENCH:BuyHold")
    if bench.empty:
        raise RuntimeError(f"Missing benchmark series for tf={tf}, symbol={sym}")

    port_al, bench_al = port.align(bench, join="inner")

    winner_metric_rows = []
    for sid in winners:
        rs = series_from_long(u["returns"], symbol=sym, cost=BASE_COST, strategy_id=sid)
        rs, _ = rs.align(bench, join="inner")
        if rs.empty:
            continue
        row_m = metrics_as_row(rs, periods_per_year=ppy)
        winner_metric_rows.append({"strategy_id": sid, **row_m})

    winner_metrics = pd.DataFrame(winner_metric_rows)
    if winner_metrics.empty:
        raise RuntimeError(f"No winner metrics computed for tf={tf}, symbol={sym}")

    best_idx = winner_metrics["Calmar"].astype(float).idxmax()
    best_single_id = str(winner_metrics.loc[best_idx, "strategy_id"])
    best_single_series = series_from_long(u["returns"], symbol=sym, cost=BASE_COST, strategy_id=best_single_id)
    best_single_series, _ = best_single_series.align(bench, join="inner")

    median_metrics = {k: float(winner_metrics[k].median()) for k in winner_metrics.columns if k != "strategy_id"}

    comparison_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "model": "Portfolio EW",
            "model_id": "PORT:EW_SPA_WINNERS",
            **metrics_as_row(port_al, periods_per_year=ppy),
        }
    )
    comparison_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "model": "Buy-and-Hold",
            "model_id": "BENCH:BuyHold",
            **metrics_as_row(bench_al, periods_per_year=ppy),
        }
    )
    comparison_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "model": "Best single (by Calmar)",
            "model_id": best_single_id,
            **metrics_as_row(best_single_series, periods_per_year=ppy),
        }
    )
    comparison_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "model": "Median SPA-winner",
            "model_id": "MEDIAN:SPA_WINNERS",
            **{k: median_metrics.get(k, np.nan) for k in METRIC_COLS},
        }
    )

    individual_vols = winners_wide.std(ddof=1)
    weighted_avg_vol = float(individual_vols.mean()) if len(individual_vols) else float("nan")
    port_vol = float(port_al.std(ddof=1)) if len(port_al) else float("nan")
    div_ratio = float(port_vol / weighted_avg_vol) if np.isfinite(weighted_avg_vol) and weighted_avg_vol > 0 else float("nan")
    avg_corr = pairwise_corr_mean(winners_wide)

    diag_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "n_winners": int(len(winners)),
            "portfolio_vol": port_vol,
            "weighted_avg_winner_vol": weighted_avg_vol,
            "div_ratio": div_ratio,
            "avg_pairwise_corr": avg_corr,
        }
    )

    case_meta_rows.append(
        {
            "timeframe": tf,
            "scenario": sc,
            "symbol": sym,
            "cost": BASE_COST,
            "best_single_id": best_single_id,
            "winners": winners,
        }
    )

portfolio_comparison = pd.DataFrame(comparison_rows).sort_values(["timeframe", "symbol", "model"])
portfolio_diag = pd.DataFrame(diag_rows).sort_values(["timeframe", "symbol"])
case_meta = pd.DataFrame(case_meta_rows).sort_values(["timeframe", "symbol"])

if portfolio_membership.shape[0] != 10:
    raise AssertionError(f"Expected 10 asset?timeframe cases, got {portfolio_membership.shape[0]}")
if portfolio_comparison.groupby(["timeframe", "symbol"]).size().nunique() != 1:
    raise AssertionError("Comparison rows per case are inconsistent.")

safe_write_parquet(portfolio_comparison, RESULTS_DIR / "portfolio_comparison_lam1_base.parquet")
safe_write_parquet(portfolio_diag, RESULTS_DIR / "portfolio_diversification_diag_lam1_base.parquet")
safe_write_parquet(case_meta, RESULTS_DIR / "portfolio_case_meta_lam1_base.parquet")

print("portfolio_comparison:", portfolio_comparison.shape)
print("portfolio_diag:", portfolio_diag.shape)
display(portfolio_comparison)
display(portfolio_diag)


portfolio_comparison: (40, 15)
portfolio_diag: (10, 9)


,timeframe,scenario,symbol,cost,model,model_id,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,n_bars
2,1d,u17,BNBUSDT,Base,Best single (by Calmar),T3:Donchian_Breakout,35.417036,1.051587,0.675672,1.371293,2.852634,-0.613364,1.714457,0.576029,1826.0
1,1d,u17,BNBUSDT,Base,Buy-and-Hold,BENCH:BuyHold,31.738836,1.008384,0.847927,1.226460,2.294844,-0.716223,1.407918,0.669708,1826.0
3,1d,u17,BNBUSDT,Base,Median SPA-winner,MEDIAN:SPA_WINNERS,2.775377,0.304154,0.446485,0.853021,1.362694,-0.567349,0.537385,0.472070,1826.0
0,1d,u17,BNBUSDT,Base,Portfolio EW,PORT:EW_SPA_WINNERS,5.345258,0.446778,0.363792,1.190919,2.239940,-0.365382,1.222769,0.226927,1826.0
6,1d,u17,BTCUSDT,Base,Best single (by Calmar),S3:Breakout_Confirm_MA,6.564356,0.444829,0.338469,1.255619,2.134935,-0.379678,1.171596,0.312308,2007.0
5,1d,u17,BTCUSDT,Base,Buy-and-Hold,BENCH:BuyHold,6.128444,0.429317,0.584867,0.902951,1.537853,-0.770678,0.557064,0.743544,2007.0
7,1d,u17,BTCUSDT,Base,Median SPA-winner,MEDIAN:SPA_WINNERS,0.265781,0.043795,0.331202,0.316650,0.460714,-0.451324,0.167300,0.422492,2007.0
4,1d,u17,BTCUSDT,Base,Portfolio EW,PORT:EW_SPA_WINNERS,1.638900,0.193003,0.205162,0.962886,1.589687,-0.340218,0.567291,0.311492,2007.0
10,1d,u17,DOGEUSDT,Base,Best single (by Calmar),R3OC:ZScore_MR_Onchain,0.843288,0.190517,0.225315,0.884545,1.504761,-0.192994,0.987162,0.126758,1280.0
9,1d,u17,DOGEUSDT,Base,Buy-and-Hold,BENCH:BuyHold,0.526538,0.128198,0.917911,0.578084,1.067467,-0.794390,0.161379,0.740897,1280.0


,timeframe,scenario,symbol,cost,n_winners,portfolio_vol,weighted_avg_winner_vol,div_ratio,avg_pairwise_corr
0,1d,u17,BNBUSDT,Base,9,0.019047,0.026270,0.725040,0.375684
1,1d,u17,BTCUSDT,Base,17,0.010741,0.016630,0.645913,0.350995
2,1d,u17,DOGEUSDT,Base,14,0.014588,0.022948,0.635674,0.299977
3,1d,u17,ETHUSDT,Base,16,0.015096,0.022933,0.658267,0.386073
4,1d,u17,XRPUSDT,Base,14,0.013855,0.023301,0.594620,0.287118
5,4h,u12,BNBUSDT,Base,12,0.008754,0.011368,0.770082,0.511134
6,4h,u12,BTCUSDT,Base,12,0.005301,0.007613,0.696281,0.409073
7,4h,u12,DOGEUSDT,Base,12,0.008437,0.012005,0.702782,0.380277
8,4h,u12,ETHUSDT,Base,12,0.007244,0.010232,0.707973,0.431407
9,4h,u12,XRPUSDT,Base,12,0.007627,0.011208,0.680493,0.361351


### Устойчивость к издержкам

In [ ]:
cost_rows = []

for r in portfolio_membership.itertuples(index=False):
    tf = str(r.timeframe)
    sc = str(r.scenario)
    sym = str(r.symbol)
    winners = list(map(str, list(r.winners)))
    ppy = int(PPY[tf])

    u = UNIVERSES[f"{tf}_{sc}"]
    n = len(winners)
    weights = np.ones(n, dtype=float) / float(n)

    for cost in COSTS:
        w = wide_returns(u["returns"], symbol=sym, cost=cost, strategy_ids=winners)
        if w.empty:
            continue
        w = w.reindex(columns=winners).fillna(0.0)
        port = (w * weights).sum(axis=1)

        cost_rows.append(
            {
                "timeframe": tf,
                "scenario": sc,
                "symbol": sym,
                "cost": cost,
                "selection_cost": BASE_COST,
                "lam_selection": float(LAMBDA),
                "n_winners": int(n),
                **metrics_as_row(port, periods_per_year=ppy),
            }
        )

portfolio_cost_sensitivity = pd.DataFrame(cost_rows).sort_values(["timeframe", "symbol", "cost"])
safe_write_parquet(portfolio_cost_sensitivity, RESULTS_DIR / "portfolio_cost_sensitivity_lam1_base_selection.parquet")

if len(portfolio_cost_sensitivity) != 30:
    logger.warning("Expected ~30 rows in cost sensitivity, got %d", len(portfolio_cost_sensitivity))

display(portfolio_cost_sensitivity)


,timeframe,scenario,symbol,cost,selection_cost,lam_selection,n_winners,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,n_bars
1,1d,u17,BNBUSDT,Base,Base,1.0,9,5.345258,0.446778,0.363792,1.190919,2.239940,-0.365382,1.222769,0.226927,1826
2,1d,u17,BNBUSDT,High,Base,1.0,9,5.129663,0.436816,0.363753,1.171995,2.203686,-0.365684,1.194515,0.234536,1826
0,1d,u17,BNBUSDT,Low,Base,1.0,9,5.413052,0.449855,0.363698,1.196978,2.251733,-0.365246,1.231650,0.223452,1826
4,1d,u17,BTCUSDT,Base,Base,1.0,17,1.638900,0.193003,0.205162,0.962886,1.589687,-0.340218,0.567291,0.311492,2007
5,1d,u17,BTCUSDT,High,Base,1.0,17,1.595771,0.189433,0.203868,0.952970,1.574750,-0.334492,0.566330,0.305570,2007
3,1d,u17,BTCUSDT,Low,Base,1.0,17,1.648118,0.193760,0.204375,0.968896,1.604583,-0.344909,0.561770,0.317662,2007
7,1d,u17,DOGEUSDT,Base,Base,1.0,14,0.556427,0.134454,0.278590,0.590747,1.017565,-0.319242,0.421165,0.269509,1280
8,1d,u17,DOGEUSDT,High,Base,1.0,14,0.539766,0.130977,0.272664,0.586366,1.015969,-0.310395,0.421970,0.270683,1280
6,1d,u17,DOGEUSDT,Low,Base,1.0,14,0.570417,0.137352,0.277980,0.600610,1.034860,-0.309340,0.444016,0.266009,1280
10,1d,u17,ETHUSDT,Base,Base,1.0,16,4.084428,0.344125,0.288336,1.169160,2.002869,-0.261786,1.314529,0.215022,2007


### Устойчивость на подпериодах

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib import colors as mcolors
    PLOTTING_AVAILABLE = True
except Exception as exc:
    PLOTTING_AVAILABLE = False
    plt = None
    mcolors = None
    logger.warning("matplotlib is unavailable; visuals will be skipped. Reason: %s", exc)

sub_rows = []
heatmap_files = []

def slice_series(s: pd.Series, start: str, end: str) -> pd.Series:
    return s.loc[(s.index >= pd.Timestamp(start)) & (s.index < pd.Timestamp(end))]

for m in case_meta.itertuples(index=False):
    tf = str(m.timeframe)
    sc = str(m.scenario)
    sym = str(m.symbol)
    ppy = int(PPY[tf])
    min_bars = int(MIN_BARS[tf])

    u = UNIVERSES[f"{tf}_{sc}"]

    port = portfolio_returns[
        (portfolio_returns["timeframe"] == tf)
        & (portfolio_returns["scenario"] == sc)
        & (portfolio_returns["symbol"] == sym)
        & (portfolio_returns["cost"] == BASE_COST)
    ].set_index("date")["ret"].astype(float)

    bench = series_from_long(u["bench"], symbol=sym, cost=BASE_COST, strategy_id="BENCH:BuyHold")
    best_single = series_from_long(u["returns"], symbol=sym, cost=BASE_COST, strategy_id=str(m.best_single_id))
    winners = list(map(str, list(m.winners)))

    port_al, bench_al = port.align(bench, join="inner")
    if PLOTTING_AVAILABLE and not port_al.empty and not bench_al.empty:
        out_eq = FIGURES_DIR / f"equity_dd_portfolio_vs_bh_{tf}_{sc}_{sym}.png"
        report_mod.plot_equity_and_drawdown(
            returns=port_al,
            bench_returns=bench_al,
            title=f"Portfolio EW vs Buy&Hold | {tf} {sc} | {sym} | Base",
            out_path=out_eq,
        )

    for label, (start, end) in SUBPERIODS.items():
        p = slice_series(port, start, end)
        b = slice_series(bench, start, end)
        s = slice_series(best_single, start, end)

        if len(p) >= min_bars:
            sub_rows.append({
                "timeframe": tf,
                "scenario": sc,
                "symbol": sym,
                "cost": BASE_COST,
                "subperiod": label,
                "model": "Portfolio EW",
                "model_id": "PORT:EW_SPA_WINNERS",
                **metrics_as_row(p, periods_per_year=ppy),
            })

        if len(b) >= min_bars:
            sub_rows.append({
                "timeframe": tf,
                "scenario": sc,
                "symbol": sym,
                "cost": BASE_COST,
                "subperiod": label,
                "model": "Buy-and-Hold",
                "model_id": "BENCH:BuyHold",
                **metrics_as_row(b, periods_per_year=ppy),
            })

        if len(s) >= min_bars:
            sub_rows.append({
                "timeframe": tf,
                "scenario": sc,
                "symbol": sym,
                "cost": BASE_COST,
                "subperiod": label,
                "model": "Best single (by Calmar)",
                "model_id": str(m.best_single_id),
                **metrics_as_row(s, periods_per_year=ppy),
            })

        winner_rows = []
        for sid in winners:
            r = series_from_long(u["returns"], symbol=sym, cost=BASE_COST, strategy_id=sid)
            rs = slice_series(r, start, end)
            if len(rs) < min_bars:
                continue
            winner_rows.append(metrics_as_row(rs, periods_per_year=ppy))

        if winner_rows:
            wdf = pd.DataFrame(winner_rows)
            med = {k: float(wdf[k].median()) for k in wdf.columns}
            sub_rows.append({
                "timeframe": tf,
                "scenario": sc,
                "symbol": sym,
                "cost": BASE_COST,
                "subperiod": label,
                "model": "Median SPA-winner",
                "model_id": "MEDIAN:SPA_WINNERS",
                **{k: med.get(k, np.nan) for k in METRIC_COLS},
            })

portfolio_subperiods = pd.DataFrame(sub_rows).sort_values(["timeframe", "symbol", "subperiod", "model"])
safe_write_parquet(portfolio_subperiods, RESULTS_DIR / "portfolio_subperiods_lam1_base.parquet")

if PLOTTING_AVAILABLE:
    for (tf, sc, sym), g in portfolio_subperiods.groupby(["timeframe", "scenario", "symbol"]):
        h = g.pivot_table(index="model", columns="subperiod", values="Calmar", aggfunc="mean")
        h = h.reindex(columns=list(SUBPERIODS.keys()))
        h = h.reindex(index=[
            "Buy-and-Hold",
            "Best single (by Calmar)",
            "Median SPA-winner",
            "Portfolio EW",
        ])

        data = h.to_numpy(dtype=float)
        finite = data[np.isfinite(data)]
        if finite.size == 0:
            continue

        vmin = min(-1.0, float(np.min(finite)), -1e-6)
        vmax = max(5.0, float(np.nanpercentile(finite, 95)))
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)

        fig_w = max(8, 1.3 * len(h.columns) + 2)
        fig_h = max(4, 0.7 * len(h.index) + 2)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))

        cmap = plt.get_cmap("RdYlGn").copy()
        cmap.set_bad(color="#d9d9d9")
        im = ax.imshow(np.ma.masked_invalid(data), aspect="auto", cmap=cmap, norm=norm)

        ax.set_xticks(np.arange(len(h.columns)))
        ax.set_yticks(np.arange(len(h.index)))
        ax.set_xticklabels(h.columns, rotation=45, ha="right")
        ax.set_yticklabels(h.index)
        ax.set_title(f"Calmar by Subperiod | {tf} {sc} | {sym}")

        for i in range(len(h.index)):
            for j in range(len(h.columns)):
                v = data[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        fig.tight_layout()

        out_h = FIGURES_DIR / f"heatmap_calmar_subperiods_{tf}_{sc}_{sym}.png"
        fig.savefig(out_h, dpi=140, bbox_inches="tight")
        plt.close(fig)
        heatmap_files.append(str(out_h))
else:
    logger.info("Skipping heatmaps and overlays because matplotlib is unavailable.")

safe_write_parquet(pd.DataFrame({"heatmap_path": heatmap_files}), RESULTS_DIR / "portfolio_heatmap_inventory.parquet")

print("portfolio_subperiods:", portfolio_subperiods.shape)
print("heatmaps saved:", len(heatmap_files))
display(portfolio_subperiods.head(30))



portfolio_subperiods: (232, 16)
heatmaps saved: 10


,timeframe,scenario,symbol,cost,subperiod,model,model_id,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,n_bars
6,1d,u17,BNBUSDT,Base,Bear 2022,Best single (by Calmar),T3:Donchian_Breakout,-0.038674,-0.038674,0.334028,0.054989,0.074558,-0.267039,-0.144827,0.267039,365.0
5,1d,u17,BNBUSDT,Base,Bear 2022,Buy-and-Hold,BENCH:BuyHold,-0.507559,-0.507559,0.726793,-0.603154,-0.918298,-0.621914,-0.816123,0.579661,365.0
7,1d,u17,BNBUSDT,Base,Bear 2022,Median SPA-winner,MEDIAN:SPA_WINNERS,-0.218024,-0.218024,0.303348,-0.255663,-0.333386,-0.332332,-0.656045,0.276599,365.0
4,1d,u17,BNBUSDT,Base,Bear 2022,Portfolio EW,PORT:EW_SPA_WINNERS,-0.180411,-0.180411,0.205911,-0.861638,-1.140770,-0.208310,-0.866068,0.189354,365.0
2,1d,u17,BNBUSDT,Base,Bull 2021,Best single (by Calmar),T3:Donchian_Breakout,19.846977,37.345949,1.360818,3.286817,8.328040,-0.368938,101.225628,0.252948,304.0
1,1d,u17,BNBUSDT,Base,Bull 2021,Buy-and-Hold,BENCH:BuyHold,12.899636,22.569810,1.593481,2.725432,5.802851,-0.616300,36.621472,0.584840,304.0
3,1d,u17,BNBUSDT,Base,Bull 2021,Median SPA-winner,MEDIAN:SPA_WINNERS,0.848226,1.090651,0.820559,1.491902,2.770385,-0.368938,3.270447,0.252948,304.0
0,1d,u17,BNBUSDT,Base,Bull 2021,Portfolio EW,PORT:EW_SPA_WINNERS,3.044131,4.352905,0.744739,2.611827,5.512106,-0.365382,11.913290,0.241345,304.0
14,1d,u17,BNBUSDT,Base,Bull 2024,Best single (by Calmar),T3:Donchian_Breakout,0.110147,0.109830,0.462841,0.450450,0.826273,-0.495017,0.221871,0.462806,366.0
13,1d,u17,BNBUSDT,Base,Bull 2024,Buy-and-Hold,BENCH:BuyHold,1.190015,1.185330,0.578596,1.637167,3.125217,-0.348643,3.399836,0.298202,366.0
